# Episode outcomes on pursuit-evasion
For each pursuer against the trained evader, over 250 episodes: the undiscounted return, the capture rate, the rate of captures off the exit, the escape rate (the evader on its exit, uncaptured, at the horizon), and the mean capture time. Every pursuer samples its move from its own distribution.

In [ ]:
%run Initialize.ipynb

## Settings and trained models

In [ ]:
SAVE_DIR = "compared"
val_episodes = 250

Context_Model = Load(TSTransformer().to(device), "encoder.pth")
critic        = Load(Critic().to(device), "critic.pth")
Player1       = Load(DefenderActor().to(device), "player1.pth")
BASELINES     = Load_Baselines(SAVE_DIR)
Player2       = Load(AttackerActor().to(device), "player2.pth")

## Rollout

In [ ]:
def Episode_Outcomes(Defender):
    defw = get_defender(Defender)
    ep_returns = []
    n_capture = n_offexit = n_escape = 0
    capture_time_sum = 0
    for i_episode in range(val_episodes):
        np.random.seed(2000 + i_episode); torch.manual_seed(2000 + i_episode); random.seed(2000 + i_episode)
        Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col = sample_config()
        defw.reset(Epi_Layout, exit_rc, Theta, Type_Theta, p_row, p_col, e_row, e_col)
        tt_t = torch.tensor(Type_Theta, dtype=torch.float32, device=device)
        m2 = init_m2(e_row, e_col)
        acc_reward = 0.0
        for t in range(max_steps):
            x = build_x(p_row, p_col, e_row, e_col)
            a_dist = defw.act_dist(x, p_row, p_col)
            with torch.no_grad():
                u_dist = Player2(torch.tensor(x, dtype=torch.float32, device=device),
                                 torch.tensor(m2, dtype=torch.float32, device=device), tt_t).cpu().numpy()
            u = int(np.random.choice(n_u, p=u_dist / u_dist.sum()))
            a = int(np.random.choice(n_a, p=a_dist))
            new_p_row, new_p_col, new_e_row, new_e_col = next_state(p_row, p_col, e_row, e_col, a, u, Epi_Layout)
            Captured = is_capture((p_row, p_col), (e_row, e_col), (new_p_row, new_p_col), (new_e_row, new_e_col))
            timeout = (t == max_steps - 1)
            acc_reward += Reward(new_p_row, new_p_col, new_e_row, new_e_col, exit_rc, timeout, Captured)
            next_m2 = m2_update(m2, e_row, e_col, u, new_e_row, new_e_col)
            y = Observation(new_e_row, new_e_col)
            defw.observe(a, u, (e_row, e_col), (new_p_row, new_p_col), (new_e_row, new_e_col), y)
            if Captured:
                n_capture += 1
                capture_time_sum += t + 1
                if (new_e_row, new_e_col) != exit_rc:
                    n_offexit += 1
                break
            if timeout:
                if (new_e_row, new_e_col) == exit_rc:
                    n_escape += 1
                break
            m2 = next_m2
            p_row, p_col, e_row, e_col = new_p_row, new_p_col, new_e_row, new_e_col
        ep_returns.append(acc_reward)
    ep_returns = np.asarray(ep_returns)
    mean_cap_time = capture_time_sum / n_capture if n_capture > 0 else float('nan')
    return (ep_returns.mean(), ep_returns.std(ddof=1) / np.sqrt(len(ep_returns)),
            n_capture / val_episodes, n_offexit / val_episodes, n_escape / val_episodes, mean_cap_time)

## Table

In [ ]:
VAL_DEFENDERS = ['Critic P1', 'Trained P1', 'RPPO', 'DR_RPPO', 'VARIBAD', 'PEARL', 'MMD', 'Random P1']
print(f"{'Pursuer':<12}{'Return':>16}{'Capture':>10}{'Off-exit':>10}{'Escape':>9}{'CapTime':>9}")
print("-" * 66)
for Defender in VAL_DEFENDERS:
    mr, mse, cr, oer, er, mct = Episode_Outcomes(Defender)
    print(f"{Defender:<12}{mr:>+10.3f}+/-{mse:<5.3f}{cr:>9.1%}{oer:>10.1%}{er:>9.1%}{mct:>9.1f}")